### 0. Mount Google Drive for Persistent Storage
**Important**: Google Colab runtime is ephemeral. Any files saved locally (like your trained model) will be **deleted** when the runtime disconnects.
To save your model persistently, mount your Google Drive and save checkpoints there.


In [ ]:
from google.colab import drive
import os

drive.mount('/content/drive')

# Create a folder for your checkpoints
save_dir = "/content/drive/MyDrive/lpcvc/2026lpcvc-track1/checkpoints"
os.makedirs(save_dir, exist_ok=True)
print(f"Checkpoints will be saved to: {save_dir}")

### Run Training (Saving to Drive)

**Performance Tip**: Reading small files from Google Drive is slow. For best performance, copy your dataset zip to `/content/` and unzip it locally before training.

```python
# 1. Copy dataset to local runtime (fast I/O)
!cp "/content/drive/MyDrive/datasets/train2017.zip" /content/
!unzip -q /content/train2017.zip -d /content/data

# 2. Run training pointing to local paths
# Assuming save_dir was set in the cell above:
!python train.py \
    --epochs 20 \
    --batch_size 128 \
    --lr 3e-4 \
    --use_distill \
    --val_every 1 \
    --save_path "/content/drive/My Drive/LPCVC_2026_Checkpoints/final_model.pth" \
    --coco_img_path "/content/data/train2017" \
    --coco_ann_path "/content/data/annotations/captions_train2017.json" \
    --flickr_root "/content/data"
```

# Optimized Training for LPCVC 2026 Track 1 in Google Colab

This notebook guides you through setting up an efficient training pipeline for the LPCVC 2026 Track 1 challenge. The goal is to maximize training throughput on Google Colab's T4/P100 GPUs by leveraging:

1.  **Hardware Acceleration**: Verifying CUDA context.
2.  **Mixed Precision (AMP)**: Speeding up math on Tensor Cores.
3.  **Efficient Data Loading**: Using multiple workers and pinned memory.
4.  **Gradient Accumulation**: Simulating larger batches for stability.
5.  **Profiling**: Identifying bottlenecks.

### Prerequisites

Clone the repository and install dependencies:

```bash
!git clone https://github.com/yourusername/2026lpcvc-track1.git
%cd 2026lpcvc-track1
!pip install -r requirements.txt
```

In [ ]:
## 1. Verify Hardware Acceleration (GPU/TPU)
# Check if CUDA is available and which device is assigned.
import torch

if torch.cuda.is_available():
    print(f"CUDA is available! Device Count: {torch.cuda.device_count()}")
    print(f"Current Device: {torch.cuda.get_device_name(0)}")
else:
    print("Warning: CUDA not found. Please enable GPU in 'Runtime' -> 'Change runtime type'.")

# Check GPU utilization
!nvidia-smi

In [ ]:
## 2. Enable Mixed Precision Training with PyTorch
# Automatic Mixed Precision (AMP) uses float16 where possible for faster ops.
# This code snippet demonstrates only the setup.

import torch
import torch.nn as nn
from track1 import XRClip, ClipLoss

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = XRClip(embed_dim=256).to(device)
criterion = ClipLoss()
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)

# Create GradScaler once
scaler = torch.amp.GradScaler('cuda')

print("GradScaler for AMP initialized.")

In [ ]:
## 3. Optimize Data Loading (num_workers & pin_memory)
# Efficient data loading is crucial to keep the GPU fed. 
# We use multiple workers and pinned memory to speed up transfer.

from torch.utils.data import DataLoader
from dataset_loader import load_flickr30k
from track1 import RetrievalDataset
from transformers import CLIPTokenizer

# Setup Sample Dataset (Small for demo)
tokenizer = CLIPTokenizer.from_pretrained("openai/clip-vit-base-patch32")
tokenizer.add_special_tokens({'cls_token': tokenizer.eos_token})

# Note: This might download data if not present
print("Loading subset of data for demonstration...")
image_paths, texts = load_flickr30k()
dataset = RetrievalDataset(image_paths[:100], texts[:100], tokenizer) # Small subset

# Speed Optimization Configuration:
# num_workers=2: Colab usually has 2 vCPUs. Using 2 workers parallels data fetching.
# pin_memory=True: Uses page-locked memory for faster Host -> Device transfer.
# prefetch_factor=2: Buffers batches ahead of time.
dataloader = DataLoader(
    dataset,
    batch_size=32,
    shuffle=True,
    num_workers=2,
    pin_memory=True,
    prefetch_factor=2,
    persistent_workers=True
)

print(f"DataLoader configured with num_workers={dataloader.num_workers}, pin_memory={dataloader.pin_memory}")

In [ ]:
## 4. Use Gradient Accumulation & Run Loop
# This cell trains the model using mixed precision and gradient accumulation.
# Gradient accumulation allows training with large effective batch sizes without high VRAM.

import torch.cuda.amp as amp
from tqdm.notebook import tqdm

accumulation_steps = 4  # Simulate 4x batch size
model.train()

# Enable TF32 for Ampere GPU (A100/A10/3090) if available
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32 = True

print("Starting training loop (AMP + Gradient Acc)...")

for epoch in range(1): # Demo 1 epoch
    optimizer.zero_grad(set_to_none=True)
    loop = tqdm(dataloader)
    
    for i, (images, texts) in enumerate(loop):
        images = images.to(device, non_blocking=True)
        texts = texts.to(device, non_blocking=True)
        
        # Mixed Precision Context
        with torch.amp.autocast('cuda'):
            img_emb, txt_emb = model(images, texts)
            loss = criterion(img_emb, txt_emb, model.logit_scale.exp())
            
            # Normalize loss for accumulation steps
            loss = loss / accumulation_steps
        
        # Scale Loss
        scaler.scale(loss).backward()
        
        if (i + 1) % accumulation_steps == 0:
            scaler.step(optimizer)
            scaler.update()
            optimizer.zero_grad(set_to_none=True)
            
        loop.set_postfix(loss=loss.item() * accumulation_steps)

print("Training cycle complete.")

In [ ]:
## 5. Profile Training Loop Performance
# Use the PyTorch profiler to measure execution time and bottlenecks.
from torch.profiler import profile, record_function, ProfilerActivity

print("Profiling the first few steps...")

with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA], record_shapes=True) as prof:
    with record_function("model_inference"):
        # Run a few steps
        optimizer.zero_grad(set_to_none=True)
        img, txt = next(iter(dataloader))
        img = img.to(device)
        txt = txt.to(device)
        
        with torch.amp.autocast('cuda'):
            out = model(img, txt)
            loss = criterion(out[0], out[1], model.logit_scale.exp())
        
        # Backward
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

print(prof.key_averages().table(sort_by="cuda_time_total", row_limit=10))